Most of the codes are just copied from other files. This jupyter notebook evaluate each model's performance on extreme pollutants.

In [1]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from loss_utils import SoftDTW2
from model_utils import GCNForecast, CosSquareFormerForecastModel
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv
from sklearn.model_selection import KFold
from train import get_train_test_data, CityDataP,CityDataForecast,create_look_ahead_mask

pollutants = [
    "pm25_median", 
    "pm10_median", 
    "o3_median", 
    "no2_median", 
    "so2_median", 
    "co_median"
]

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6378.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2.0)**2
         + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2))
         * math.sin(dlon / 2.0)**2)
    c = 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))
    return R * c



In [2]:
df = pd.read_csv("city_pollution_data2.csv")
train_set, test_set = get_train_test_data(df,method = "mean")

cities_list = list(train_set.keys())

all_train = pd.DataFrame()
for city in cities_list:
  all_train = all_train._append(train_set[city], ignore_index=True)

all_test = pd.DataFrame({})
for city in test_set:
  all_test = all_test._append(test_set[city], ignore_index=True)

concat_df = pd.concat([all_train,all_test],axis=0)

col_max = {}
col_mean = {}
col_mean2 = {}
col_std = {}
normalization_type = 'mean_std'  # 'mean_std' or 'max'
DROP_ONEHOT = True  # whether to drop one-hot columns
pollutants = ["pm25_median","pm10_median", "o3_median", "so2_median", "no2_median", "co_median"]

####### Deleting incomplete rows, comment these out if using imputation
'''
for city in cities_list:
    print(f"Training set {city} shape: {train_set[city].shape}")
    print(f"Testing set {city} shape: {test_set[city].shape}\n")
    train_set[city].dropna(subset = pollutants, inplace=True)
    test_set[city].dropna(subset = pollutants, inplace=True)
    print(f"Training set {city} shape: {train_set[city].shape}")
    print(f"Testing set {city} shape: {test_set[city].shape}")
'''


for city in cities_list:
  col_mean[city] = {}
  #print(train_set[city].columns)
  train_set[city] = train_set[city].drop(['index', 'Date', 'City'], axis=1)
  test_set[city] = test_set[city].drop(['index', 'Date', 'City'], axis=1)


  for col in train_set[city]:
    #if col in ["index", "Date", "City"]:
      #continue

    train_set[city][col] = train_set[city][col].astype("float")
    test_set[city][col] = test_set[city][col].astype("float")


    if col in ["pm25_median","pm10_median", "o3_median", "so2_median", "no2_median", "co_median"]:
      ###################
      _mean = np.nanmean(train_set[city][col])
      if np.isnan(_mean):
        _mean = 0
      
      col_mean[city][col] = _mean
      train_set[city][col] = train_set[city][col].fillna(_mean)
      test_set[city][col] = test_set[city][col].fillna(_mean)

    if normalization_type == 'mean_std':
      col_mean2[col] = np.nanmean(concat_df[col].astype("float"))
      col_std[col] = np.nanstd(concat_df[col].astype("float"))
      train_set[city][col] = (train_set[city][col] - col_mean2[col]) / (col_std[col] + 0.001)
      test_set[city][col] = (test_set[city][col] - col_mean2[col]) / (col_std[col] + 0.001)

    else:
      col_max[col] = concat_df[col].astype("float").max()
      train_set[city][col] = train_set[city][col] / (col_max[col] + 0.001)
      test_set[city][col] = test_set[city][col] / (col_max[col] + 0.001)

  if DROP_ONEHOT:
    train_set[city].drop(train_set[city].columns[-19:], axis=1, inplace=True)
    test_set[city].drop(test_set[city].columns[-19:], axis=1, inplace=True)



# City Coordinates (54 Cities)
city_coords = {
    'philadelphia': (39.9526, -75.1652),
    'columbus': (39.9612, -82.9988),
    'providence': (41.8240, -71.4128),
    'oklahoma city': (35.4676, -97.5164),
    'dallas': (32.7767, -96.7970),
    'miami': (25.7617, -80.1918),
    'raleigh': (35.7796, -78.6382),
    'staten island': (40.5795, -74.1502),
    'hartford': (41.7658, -72.6734),
    'atlanta': (33.7490, -84.3880),
    'boise': (43.6150, -116.2023),
    'detroit': (42.3314, -83.0458),
    'seattle': (47.6062, -122.3321),
    'saint paul': (44.9537, -93.0900),
    'las vegas': (36.1699, -115.1398),
    'san antonio': (29.4241, -98.4936),
    'memphis': (35.1495, -90.0490),
    'san francisco': (37.7749, -122.4194),
    'springfield': (37.2089, -93.2923),
    'baltimore': (39.2904, -76.6122),
    'portland': (45.5051, -122.6750),
    'salt lake city': (40.7608, -111.8910),
    'albuquerque': (35.0844, -106.6504),
    'tucson': (32.2226, -110.9747),
    'jacksonville': (30.3322, -81.6557),
    'sacramento': (38.5816, -121.4944),
    'madison': (43.0731, -89.4012),
    'columbia': (34.0007, -81.0348),
    'indianapolis': (39.7684, -86.1581),
    'los angeles': (34.0522, -118.2437),
    'manhattan': (40.7831, -73.9712),
    'tallahassee': (30.4383, -84.2807),
    'milwaukee': (43.0389, -87.9065),
    'honolulu': (21.3069, -157.8583),
    'richmond': (37.5407, -77.4360),
    'austin': (30.2672, -97.7431),
    'el paso': (31.7619, -106.4850),
    'fort worth': (32.7555, -97.3308),
    'salem': (44.9429, -123.0351),
    'chicago': (41.8781, -87.6298),
    'boston': (42.3601, -71.0589),
    'houston': (29.7604, -95.3698),
    'denver': (39.7392, -104.9903),
    'oakland': (37.8044, -122.2711),
    'phoenix': (33.4484, -112.0740),
    'nashville': (36.1627, -86.7816),
    'omaha': (41.2565, -95.9345),
    'jackson': (32.2988, -90.1848),
    'little rock': (34.7465, -92.2896),
    'fresno': (36.7378, -119.7871),
    'san diego': (32.7157, -117.1611),
    'charlotte': (35.2271, -80.8431),
    'brooklyn': (40.6782, -73.9442),
    'san jose': (37.3382, -121.8863),
}
city_names = list(city_coords.keys())
assert len(city_names) == 54, f"Expected 54 cities, got {len(city_names)}"

# Build Graph Based on Haversine Distance < 400 km
DISTANCE_THRESHOLD = 400

edge_index_list = [[], []]
num_nodes = len(city_names)

for i in range(num_nodes):
    for j in range(i+1, num_nodes):
        lat1, lon1 = city_coords[city_names[i]]
        lat2, lon2 = city_coords[city_names[j]]
        dist = haversine_distance(lat1, lon1, lat2, lon2)
        if dist < DISTANCE_THRESHOLD:
            # undirected edge => both directions
            edge_index_list[0].append(i)
            edge_index_list[1].append(j)
            edge_index_list[0].append(j)
            edge_index_list[1].append(i)

edge_index = torch.tensor(edge_index_list, dtype=torch.long)
print(f"Graph built: {num_nodes} nodes, {edge_index.size(1)//2} undirected edges.")


# Prepare Data from train_set & test_set
PAST_DAYS = 14
FUTURE_DAYS = 7


###################################
'''
for city in cities_list:
   if len(train_set[city]) < PAST_DAYS + FUTURE_DAYS:
      del train_set[city]

for city in cities_list:
   if len(test_set[city]) < PAST_DAYS + FUTURE_DAYS:
      del test_set[city]
'''

# Check minimum length
min_train_days = min(len(df) for df in train_set.values())
min_test_days  = min(len(df) for df in test_set.values())
if min_train_days < PAST_DAYS + FUTURE_DAYS:
    raise ValueError("Not enough training data for the specified window sizes.")
if min_test_days < PAST_DAYS + FUTURE_DAYS:
    raise ValueError("Not enough test data for the specified window sizes.")


Graph built: 54 nodes, 81 undirected edges.


In [3]:
'''
def build_window_data(city_dict, min_days, features, SELECTED_COLUMN, top_percentile=None):
    data_list = []
    total_days = min_days

    # First, collect all possible y_window values
    all_y_max_values = []

    for start in range(total_days - (PAST_DAYS + FUTURE_DAYS) + 1):
        for city in city_names:
            df = city_dict[city]
            y_window = df.iloc[start+PAST_DAYS : start+PAST_DAYS+FUTURE_DAYS][SELECTED_COLUMN].values
            all_y_max_values.append(np.max(y_window))  # or np.mean()

    threshold = np.percentile(all_y_max_values, 80) if top_percentile else None

    for start in range(total_days - (PAST_DAYS + FUTURE_DAYS) + 1):
        features_all = []
        labels_all = []
        high_window = False

        for city in city_names:
            df = city_dict[city]
            x_window = df.iloc[start:start+PAST_DAYS][features].values
            x_flat = x_window.flatten()

            y_window = df.iloc[start+PAST_DAYS : start+PAST_DAYS+FUTURE_DAYS][SELECTED_COLUMN].values
            labels_all.append(y_window)
            features_all.append(x_flat)

            if top_percentile and np.max(y_window) >= threshold:
                high_window = True  # keep this window

        if top_percentile is None or high_window:
            x_tensor = torch.tensor(np.stack(features_all), dtype=torch.float)
            y_tensor = torch.tensor(np.stack(labels_all), dtype=torch.float)
            data = Data(x=x_tensor, y=y_tensor, edge_index=edge_index)
            data_list.append(data)

    return data_list
'''

def build_window_data(city_dict, min_days, features, SELECTED_COLUMN, top_percentile=None):
    window_records = []  # Will store tuples: (score, features_all, labels_all)

    total_days = min_days
    for start in range(total_days - (PAST_DAYS + FUTURE_DAYS) + 1):
        features_all = []
        labels_all = []
        scores = []
        for city in city_names:
            df = city_dict[city]
            x_window = df.iloc[start:start+PAST_DAYS][features].values
            x_flat = x_window.flatten()
            y_window = df.iloc[start+PAST_DAYS : start+PAST_DAYS+FUTURE_DAYS][SELECTED_COLUMN].values

            features_all.append(x_flat)
            labels_all.append(y_window)
            # Use mean or max as score
            scores.append(np.max(y_window))

        # For multi-city, aggregate score as mean/max across cities
        score = np.max(scores)
        window_records.append((score, features_all, labels_all))

    # Now select top-percentile
    if top_percentile:
        # Sort windows by score descending
        window_records.sort(key=lambda x: x[0], reverse=True)
        n_keep = int(len(window_records) * (100 - top_percentile) / 100)
        window_records = window_records[:n_keep]

    data_list = []
    for score, features_all, labels_all in window_records:
        x_tensor = torch.tensor(np.stack(features_all), dtype=torch.float)
        y_tensor = torch.tensor(np.stack(labels_all), dtype=torch.float)
        data = Data(x=x_tensor, y=y_tensor, edge_index=edge_index)
        data_list.append(data)
    return data_list


In [4]:
for SELECTED_COLUMN in pollutants:
    print(f"\n{'=' * 50}")
    print(f"Processing Selected Column: {SELECTED_COLUMN}")
    print(f"{'=' * 50}\n")

    features = ['Population Staying at Home', 'Population Not Staying at Home',
                'mil_miles', 'pressure_median', SELECTED_COLUMN, 'humidity_median',
                'temperature_median', 'dew_median', 'wind-speed_median',
                'wind-gust_median', 'pp_feat']

    #train_data_list = build_window_data(train_set, min_train_days, features, SELECTED_COLUMN)
    test_data_list = build_window_data(test_set, min_test_days, features, SELECTED_COLUMN,top_percentile = 80)
    print(f"Prepared {len(test_data_list)} testing windows.\n")
    test_loader = DataLoader(test_data_list, batch_size=16, shuffle=False)

    '''
    for hch in h_channels_list:
        for lr in lr_list:
            fold_rmses = []
            fold_mapes = []
            for fold_i, (train_idx, val_idx) in enumerate(kf.split(all_indices)):
                # Prepare fold train/val subsets
                fold_train_list = [train_data_list[i] for i in train_idx]
                fold_val_list   = [train_data_list[i] for i in val_idx]

                train_loader = DataLoader(fold_train_list, batch_size=16, shuffle=True)
                val_loader   = DataLoader(fold_val_list, batch_size=16, shuffle=False)

                device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
                model  = GCNForecast(in_channels=PAST_DAYS*11, hidden_channels=hch, out_channels=FUTURE_DAYS).to(device)
                optimizer = optim.Adam(model.parameters(), lr=lr)

                num_epochs = 5  # can adjust if needed
                for epoch in range(num_epochs):
                    model.train()
                    for batch in train_loader:
                        batch = batch.to(device)
                        optimizer.zero_grad()
                        out = model(batch.x, batch.edge_index)  # shape: [54, 7]
                        loss_val = combined_loss(out, batch.y)
                        loss_val.backward()
                        optimizer.step()
    '''

    if SELECTED_COLUMN in ["pm25_median","pm10_median", "so2_median", "no2_median"]:
        h = 128
    elif SELECTED_COLUMN in ["o3_median"]:
        h = 64
    else:
        h = 32
    model = GCNForecast(in_channels=PAST_DAYS*11, hidden_channels=h, out_channels=FUTURE_DAYS)
    model.load_state_dict(torch.load(f"GCN_{SELECTED_COLUMN}.pth"))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    # 
    val_preds = []
    val_truth = []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index)
            val_preds.append(out.cpu().numpy())
            val_truth.append(batch.y.cpu().numpy())
    val_preds = np.concatenate(val_preds, axis=0)
    val_truth = np.concatenate(val_truth, axis=0)

    val_truth = val_truth * col_std[SELECTED_COLUMN] + col_mean2[SELECTED_COLUMN]
    val_preds = val_preds * col_std[SELECTED_COLUMN] + col_mean2[SELECTED_COLUMN]

    rmse = np.sqrt(np.mean((val_preds - val_truth)**2))
    mape = np.mean(np.abs((val_preds - val_truth) / (val_truth + 1e-4))) * 100

    print(f"{SELECTED_COLUMN} Extreme RMSE: {rmse:.4f}")


Processing Selected Column: pm25_median

Prepared 8 testing windows.

pm25_median Extreme RMSE: 17.7159

Processing Selected Column: pm10_median

Prepared 8 testing windows.

pm10_median Extreme RMSE: 10.5692

Processing Selected Column: o3_median

Prepared 8 testing windows.

o3_median Extreme RMSE: 8.9026

Processing Selected Column: so2_median

Prepared 8 testing windows.

so2_median Extreme RMSE: 2.1427

Processing Selected Column: no2_median

Prepared 8 testing windows.

no2_median Extreme RMSE: 4.5606

Processing Selected Column: co_median

Prepared 8 testing windows.

co_median Extreme RMSE: 5.4692


In [5]:
class CityDataP(torch.utils.data.Dataset):
  def __init__(self, selected_column, split, top_percentile=False):
    self.split = split
    self.selected_column = selected_column
    self.dataset = train_set if split == "train" else test_set
    self.input_seq_len = 14
    self.target_seq_len = 7
    self.total_seq = self.input_seq_len + self.target_seq_len
    self.indices = []

    if split == "test" and top_percentile:
      all_targets = []

      # Collect all target values
      for city in cities_list:
        df = self.dataset[city]
        for i in range(len(df) - self.total_seq + 1):
          target_seq = df.iloc[i + self.input_seq_len: i + self.total_seq][selected_column].values
          all_targets.append(np.max(target_seq))

      threshold = np.percentile(all_targets, 80)

      # Save indices where target exceeds threshold
      for city in cities_list:
        df = self.dataset[city]
        for i in range(len(df) - self.total_seq + 1):
          target_seq = df.iloc[i + self.input_seq_len: i + self.total_seq][selected_column].values
          if np.max(target_seq) >= threshold:
            self.indices.append((city, i))
    else:
      for city in cities_list:
        df = self.dataset[city]
        for i in range(len(df) - self.total_seq + 1):
          self.indices.append((city, i))

  def __len__(self):
    return len(self.indices)

  def __getitem__(self, idx):
    city, start_idx = self.indices[idx]
    df = self.dataset[city]
    window = df.iloc[start_idx : start_idx + self.total_seq].copy()
    window = window.drop(['index', 'Date', 'City'], axis=1)

    Y_all = pd.DataFrame({self.selected_column: window[self.selected_column]})
    #features = window.drop(columns=[self.selected_column], errors='ignore')
    features = window.copy()

    for col in features.columns.tolist():
      if col == self.selected_column:
        continue
      elif col in ["pm25_median", "pm10_median", "o3_median", "so2_median", "no2_median", "co_median"]:
        features.drop(col, axis=1, inplace=True)
      else:
        features[col] = features[col].astype("float")

    X_input = features.iloc[:self.input_seq_len, :]
    Y_target = Y_all.iloc[self.input_seq_len:self.input_seq_len+self.target_seq_len, :]

    return torch.tensor(X_input.values, dtype=torch.float32), \
       torch.tensor(Y_target.values, dtype=torch.float32), \
       torch.tensor(Y_all.values, dtype=torch.float32)



In [6]:


device = 'cpu'

import warnings
warnings.filterwarnings('ignore')

# Data Pre-processing

df = pd.read_csv("city_pollution_data2.csv")

DROP_ONEHOT = True
SEQ_LENGTH = 7

if DROP_ONEHOT:
  INPUT_DIM = 10 
else:
  INPUT_DIM = 29

HIDDEN_DIM = 32
LAYER_DIM = 3


normalization_type = 'mean_std' # 'max', mean_std

train_set, test_set = get_train_test_data(df)

cities_list = list(train_set.keys())

all_train = pd.DataFrame()
for city in cities_list:
  all_train = all_train._append(train_set[city], ignore_index=True)

all_test = pd.DataFrame({})
for city in test_set:
  all_test = all_test._append(test_set[city], ignore_index=True)

concat_df = pd.concat([all_train,all_test],axis=0)

# ---------------------------------------------------------------------------- #
col_max = {}
col_mean = {}
col_mean2 = {}
col_std = {}


for city in cities_list:
  col_mean[city] = {}
  for col in train_set[city]:
    if col in ["index", "Date", "City"]:
      continue

    train_set[city][col] = train_set[city][col].astype("float")
    test_set[city][col] = test_set[city][col].astype("float")

    if col in ["pm25_median", "o3_median", "so2_median", "no2_median", "pm10_median", "co_median"]:
      _mean = np.nanmean(train_set[city][col])
      if np.isnan(_mean):
        _mean = 0
      col_mean[city][col] = _mean
      train_set[city][col] = train_set[city][col].fillna(_mean)
      test_set[city][col] = test_set[city][col].fillna(_mean)

    if normalization_type == 'mean_std':
      col_mean2[col] = np.nanmean(concat_df[col].astype("float"))
      col_std[col] = np.nanstd(concat_df[col].astype("float"))
      train_set[city][col] = (train_set[city][col] - col_mean2[col]) / (col_std[col] + 0.001)
      test_set[city][col] = (test_set[city][col] - col_mean2[col]) / (col_std[col] + 0.001)

    else:
      col_max[col] = concat_df[col].astype("float").max()
      train_set[city][col] = train_set[city][col] / (col_max[col] + 0.001)
      test_set[city][col] = test_set[city][col] / (col_max[col] + 0.001)

  if DROP_ONEHOT:
    train_set[city].drop(train_set[city].columns[-19:], axis=1, inplace=True)
    test_set[city].drop(test_set[city].columns[-19:], axis=1, inplace=True)


for SELECTED_COLUMN in pollutants:
    #print("111111111111111111111111111111111111111111111111")
    train_data = CityDataP(SELECTED_COLUMN, "train")
    val_data = CityDataP(SELECTED_COLUMN, "test",top_percentile = True)
    print("val_data size:", len(val_data))

    sampleLoader = DataLoader(train_data, 32, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_data, 4096, shuffle=False, num_workers=0)
    #print("222222222222222222222222222222222222222222222222")
    model = CosSquareFormerForecastModel(input_dim=11,D=32,H=4,N_layers=4,ff_dim=128,dropout=0.3,max_seq_len=64).to(device)
    model.load_state_dict(torch.load(f"CSF14-7_{SELECTED_COLUMN}.pth"))
    #print("4444444444444444444444444")
    model.eval()
    #print("5555555555555")
    mse_list = []
    total_se = 0.0
    total_pe = 0.0
    total_valid = 0.0

    for x_val, y_val, _ in val_loader:
        #print("3333333333333333333333333333333333333333333333333333333")
        x_val, y_val = [t.to(device).float() for t in (x_val, y_val)]
        mask = create_look_ahead_mask(x_val.shape[1])
        out, _ = model(x_val, mask)
              #print(out.shape, y_val.shape)
        ytrue = y_val.squeeze(-1).cpu().numpy()
        ypred = out.squeeze(-1).cpu().detach().numpy()
        ytrue = ytrue.ravel()
        ypred = ypred.ravel()

        true_valid = np.isnan(ytrue) != 1
        ytrue = ytrue[true_valid]
        ypred = ypred[true_valid]

        if normalization_type == 'mean_std':
            ytrue = (ytrue * col_std[SELECTED_COLUMN]) + col_mean2[SELECTED_COLUMN]
            ypred = (ypred * col_std[SELECTED_COLUMN]) + col_mean2[SELECTED_COLUMN]
        else:
            ytrue = (ytrue * col_max[SELECTED_COLUMN])
            ypred = (ypred * col_max[SELECTED_COLUMN])
              
        se = (ytrue - ypred)**2
        pe = np.abs((ytrue - ypred) / (ytrue + 1e-4))
              
        total_se += np.sum(se)
        total_pe += np.sum(pe)
        total_valid += len(ytrue)

    eval_mse = total_se / total_valid
    eval_mape = total_pe / total_valid

    print(f"{SELECTED_COLUMN} Extreme RMSE: {np.sqrt(eval_mse):.4f}")


val_data size: 476
pm25_median Extreme RMSE: 21.8909
val_data size: 483
pm10_median Extreme RMSE: 10.5522
val_data size: 461
o3_median Extreme RMSE: 10.1826
val_data size: 444
so2_median Extreme RMSE: 2.4693
val_data size: 435
no2_median Extreme RMSE: 6.4974
val_data size: 481
co_median Extreme RMSE: 10.1584


In [7]:

import numpy as np
np.float_ = np.float64
import pandas as pd
import random
from datetime import datetime
from sklearn.metrics import mean_squared_error
from skfda.representation.grid import FDataGrid
from skfda.ml.regression import LinearRegression
from skfda.representation.basis import BSpline
from skfda.representation.basis import BSplineBasis, VectorValuedBasis
# ---------------- Constants ----------------
exogenous_cols = [
    "Population Staying at Home", "Population Not Staying at Home", "humidity_median",
    "temperature_median", "dew_median", "wind-speed_median", "mil_miles",
    "wind-gust_median", "pressure_median", "pp_feat"
]
pollutants = ["pm25_median", "pm10_median", "o3_median", "so2_median", "no2_median", "co_median"]

# ---------------- Helper Function ----------------
def get_train_test_data(df):
    df["Population Staying at Home"] = df["Population Staying at Home"].str.replace(",", "")
    df["Population Not Staying at Home"] = df["Population Not Staying at Home"].str.replace(",", "")
    df["weekday"] = df["Date"].apply(lambda x: datetime.strptime(x, "%Y-%m-%d").weekday())
    df["month"] = df["Date"].apply(lambda x: datetime.strptime(x, "%Y-%m-%d").month - 1)
    df = df.join(pd.get_dummies(df.pop('weekday'), prefix='day'))
    df = df.join(pd.get_dummies(df.pop('month'), prefix='month'))

    for col in df.columns:
        for x in ["min", "max", "count", "County", "past_week", "latitude", "longitude", "State", "variance"]:
            if x in col:
                df.drop([col], axis=1, inplace=True)

    cities_list = list(set(df['City']))
    city_df, test_set, train_set = {}, {}, {}
    TEST_SET_SIZE = 60

    for city in cities_list:
        city_df[city] = df[df['City'] == city].sort_values('Date').reset_index()
        for col in city_df[city].columns:
            if col in pollutants:
                continue
            try:
                _mean = np.nanmean(city_df[city][col].astype(float))
                #city_df[city][col].fillna(_mean if not np.isnan(_mean) else 0, inplace=True)
                city_df[city][col] = city_df[city][col].fillna(_mean if not np.isnan(_mean) else 0)

            except:
                pass

        random.seed(0)
        if city_df[city].shape[0] < TEST_SET_SIZE + 21:
            continue
        start = random.randint(0, city_df[city].shape[0] - TEST_SET_SIZE)
        test_set[city] = city_df[city].iloc[start:start + TEST_SET_SIZE]
        train_set[city] = city_df[city].drop(index=list(range(start, start + TEST_SET_SIZE)))

    return train_set, test_set

# ---------------- Load Data ----------------
df = pd.read_csv("city_pollution_data2.csv")
train_set, test_set = get_train_test_data(df)
cities_list = list(train_set.keys())
all_train = pd.concat([train_set[city] for city in cities_list])
all_test = pd.concat([test_set[city] for city in cities_list])
concat_df = pd.concat([all_train, all_test])

col_mean2 = {col: np.nanmean(concat_df[col].astype("float")) for col in pollutants}
col_std = {col: np.nanstd(concat_df[col].astype("float")) for col in pollutants}

# ---------------- FDA Model ----------------
print("\n--- FDA RMSE Results ---")
for pollutant in pollutants:
    feature_cols = exogenous_cols + [pollutant]
    X_all, Y_all = [], []

    for city in cities_list:
        df_train = train_set[city].copy()
        df_train[pollutant] = df_train[pollutant].astype(float)
        #df_train[pollutant].fillna(df_train[pollutant].mean(), inplace=True)
        df_train[pollutant] = df_train[pollutant].fillna(df_train[pollutant].mean())

        data = df_train[feature_cols].values
        series = df_train[pollutant].values

        for i in range(len(data) - 14 - 7):
            #x_seq = data[i:i+14].as        # shape: (14, 11)
            #y_seq = series[i+14:i+21]
            x_seq = data[i:i+14].astype(float)
            y_seq = series[i+14:i+21].astype(float)
            if not (np.any(np.isnan(x_seq)) or np.any(np.isnan(y_seq))):
                y_seq = (y_seq - col_mean2[pollutant]) / (col_std[pollutant] + 1e-3)
                X_all.append(x_seq)  # (14, 11)
                Y_all.append(y_seq)

    if len(X_all) == 0:
        continue

    X_all = np.array(X_all)  # shape: (N, 14, 11)
    Y_all = np.array(Y_all)

    #basis = BSpline(domain_range=(0, 14), n_basis=7)
    basis = BSplineBasis(domain_range=(0, 14), n_basis=7)
    vector_basis = VectorValuedBasis([basis] * 11)
    fd_X = FDataGrid(data_matrix=X_all, grid_points=np.arange(14))

    fd_basis_X = fd_X.to_basis(vector_basis)

    models = []
    for i in range(7):
        model = LinearRegression()
        model.fit(fd_basis_X, Y_all[:, i])
        models.append(model)

# ---------------- Evaluation ----------------
    window_scores = []
    window_X = []
    window_Y = []

    for city in cities_list:
        df_test = test_set[city].copy()
        df_test[pollutant] = df_test[pollutant].astype(float)
        df_test[pollutant] = df_test[pollutant].fillna(df_test[pollutant].mean())

        data = df_test[feature_cols].values
        series = df_test[pollutant].values

        for i in range(len(data) - 14 - 7):
            x_seq = data[i:i+14].astype(np.float64)
            y_seq = series[i+14:i+21].astype(np.float64)
            if not (np.any(np.isnan(x_seq)) or np.any(np.isnan(y_seq))):
                score = np.max(y_seq)
                y_seq_norm = (y_seq - col_mean2[pollutant]) / (col_std[pollutant] + 1e-3)
                window_scores.append(score)
                window_X.append(x_seq)
                window_Y.append(y_seq_norm)

    # Filter top 20% windows based on pollutant max value
    threshold = np.percentile(window_scores, 80)
    X_test = [x for x, s in zip(window_X, window_scores) if s > threshold]
    Y_test = [y for y, s in zip(window_Y, window_scores) if s > threshold]

    if len(X_test) == 0:
        continue

    X_test = np.array(X_test)
    Y_test = np.array(Y_test)
    fd_X_test = FDataGrid(data_matrix=X_test, grid_points=np.arange(14))
    fd_basis_X_test = fd_X_test.to_basis(vector_basis)

    Y_pred = [model.predict(fd_basis_X_test).reshape(-1, 1) for model in models]
    Y_pred = np.hstack(Y_pred)

    # denormalize
    mean, std = col_mean2[pollutant], col_std[pollutant]
    Y_test_denorm = Y_test * std + mean
    Y_pred_denorm = Y_pred * std + mean

    rmse = np.sqrt(mean_squared_error(Y_test_denorm, Y_pred_denorm))
    print(f"{pollutant}: RMSE = {rmse:.4f}")



--- FDA RMSE Results ---
pm25_median: RMSE = 20.3153
pm10_median: RMSE = 12.9575
o3_median: RMSE = 8.4232
so2_median: RMSE = 2.4741
no2_median: RMSE = 6.4489
co_median: RMSE = 9.3938


In [17]:
import pandas as pd
import numpy as np
import matplotlib as plt
from datetime import datetime
import random

from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score

from train import get_train_test_data
from joblib import dump
from joblib import load

def root_mean_squared_error(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Load Data
df = pd.read_csv("city_pollution_data2.csv")
train_set_dict, test_set_dict = get_train_test_data(df)

train_frames = []
test_frames = []
for city in train_set_dict.keys():
    train_frames.append(train_set_dict[city])
    test_frames.append(test_set_dict[city])

df_train_all = pd.concat(train_frames, axis=0).reset_index(drop=True)
df_test_all  = pd.concat(test_frames, axis=0).reset_index(drop=True)

# Pollutants & parameters
pollutants = [
    "pm25_median", 
    "pm10_median", 
    "o3_median", 
    "no2_median", 
    "so2_median", 
    "co_median"
]

HORIZON   = 7    # Next 7 days
LAG_DAYS  = 14   # Past 14 days input

# Create 14-lag + 7-future-day columns
def create_14lag_7future(df, pollutant,top_percentile = False):
    df = df.sort_values(["City", "Date"]).copy()
    # 14-lag features
    for i in range(1, LAG_DAYS + 1):
        df[f"{pollutant}_lag_{i}"] = df.groupby("City")[pollutant].shift(i)
    # 7 future days as targets
    for d in range(1, HORIZON + 1):
        df[f"{pollutant}_target_{d}"] = df.groupby("City")[pollutant].shift(-d)

    if top_percentile:
        target_cols = [f"{pollutant}_target_{d}" for d in range(1, HORIZON + 1)]
        df["max_future"] = df[target_cols].max(axis=1)
        
        # Compute 80th percentile threshold
        threshold = df["max_future"].quantile(0.80)

        # Filter to only keep rows where future max >= threshold
        df = df[df["max_future"] >= threshold].drop(columns=["max_future"])
    return df


exogenous_cols = [
    "Population Staying at Home",
    "Population Not Staying at Home",
    "humidity_median",
    "temperature_median",
    "dew_median",
    "wind-speed_median",
    "mil_miles",
    "wind-gust_median",
    "pressure_median",
    "pp_feat"
]


# Missing Data Approaches
def drop_incomplete_rows(df, needed_cols):
    return df.dropna(subset=needed_cols+pollutants)

def impute_mean(df, needed_cols):
    for c in needed_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')
        mean = np.nanmean(df[c])
        df[c] = df[c].fillna(mean)
    return df

def impute_median(df, needed_cols):
    for c in needed_cols:
        df[c] = df[c].fillna(df[c].median())
    return df

for p in pollutants:
    df_train_p = create_14lag_7future(df_train_all, p)
    df_test_p  = create_14lag_7future(df_test_all, p,top_percentile = True)

    lag_cols    = [f"{p}_lag_{i}" for i in range(1, LAG_DAYS+1)]
    target_cols = [f"{p}_target_{d}" for d in range(1, HORIZON+1)]
    needed_cols = exogenous_cols + lag_cols + target_cols

    # Apply missing-data strategy

    print("train before drop:",df_train_p.shape)
    df_train_p = impute_mean(df_train_p, needed_cols)
    df_train_p = df_train_p.dropna(subset=needed_cols)
    print("train after drop:",df_train_p.shape)

    df_test_p = impute_mean(df_test_p, needed_cols)
    df_test_p = df_test_p.dropna(subset=needed_cols)

    # X, Y for train & test
    X_train = df_train_p[exogenous_cols + lag_cols]
    Y_train = df_train_p[target_cols]

    X_test  = df_test_p[exogenous_cols + lag_cols]
    Y_test  = df_test_p[target_cols]

    print("Train shape:", X_train.shape, Y_train.shape, 
          "Test shape:", X_test.shape,  Y_test.shape)
    # Scale X
    scaler_X = StandardScaler()
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled  = scaler_X.transform(X_test)

    # Scale Y
    scaler_Y = StandardScaler()
    Y_train_scaled = scaler_Y.fit_transform(Y_train)

    # MultiOutput SVR: 7-day forecast
    model = load(f'SVR_{p}.joblib')
    Y_pred_scaled = model.predict(X_test_scaled)
    Y_pred = scaler_Y.inverse_transform(Y_pred_scaled)

    rmse_list = []
    for day_idx in range(HORIZON):
        true_day = Y_test.iloc[:, day_idx].values  # e.g. p_target_1
        pred_day = Y_pred[:, day_idx]
        day_rmse = root_mean_squared_error(true_day, pred_day)
        rmse_list.append(day_rmse)

    avg_rmse = np.mean(rmse_list)
    print(f"{p} RMSE = {avg_rmse:.3f}")


train before drop: (32356, 59)
train after drop: (32356, 59)
Train shape: (32356, 24) (32356, 7) Test shape: (705, 24) (705, 7)
pm25_median RMSE = 21.768
train before drop: (32356, 59)
train after drop: (32356, 59)
Train shape: (32356, 24) (32356, 7) Test shape: (333, 24) (333, 7)
pm10_median RMSE = 15.176
train before drop: (32356, 59)
train after drop: (32356, 59)
Train shape: (32356, 24) (32356, 7) Test shape: (693, 24) (693, 7)
o3_median RMSE = 8.064
train before drop: (32356, 59)
train after drop: (32356, 59)
Train shape: (32356, 24) (32356, 7) Test shape: (451, 24) (451, 7)
no2_median RMSE = 6.884
train before drop: (32356, 59)
train after drop: (32356, 59)
Train shape: (32356, 24) (32356, 7) Test shape: (1099, 24) (1099, 7)
so2_median RMSE = 1.961
train before drop: (32356, 59)
train after drop: (32356, 59)
Train shape: (32356, 24) (32356, 7) Test shape: (469, 24) (469, 7)
co_median RMSE = 9.412
